In [0]:
# R2 — Chunk, embed, index

# Chunks the 8 policy markdown documents into `##` sections and loads them into `claims.genai.policy_chunks` (Delta, Change Data Feed enabled) with an idempotent MERGE keyed on `chunk_id` — run twice, the second load must insert nothing and delete nothing. Ends with the Delta history proof. Vector Search index creation is done after user review.

In [0]:
# Paper 4 config cell - change endpoint names HERE and nowhere else
CATALOG = "claims"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"
EMBED_MODEL = "databricks-gte-large-en"

print(f"CATALOG={CATALOG} | CHAT_MODEL={CHAT_MODEL} | EMBED_MODEL={EMBED_MODEL}")

In [0]:
%sql
-- R2 table: policy_chunks - one row per ## section of each policy document
-- delta.enableChangeDataFeed = true so the Vector Search index can follow the table
CREATE TABLE if not exists claims.genai.policy_chunks (
  chunk_id    STRING NOT NULL,
  doc_name    STRING,
  heading     STRING,
  chunk_index INT,
  text        STRING
)
TBLPROPERTIES (delta.enableChangeDataFeed = true);



In [0]:
%sql
SHOW TBLPROPERTIES claims.genai.policy_chunks;

In [0]:
%sql
-- R2 loader: rebuild all chunks from the binder, then MERGE keyed on chunk_id
-- chunk_id = <filename>#<anchor>; anchor = heading lowercased, runs of non-alphanumerics -> single '-', leading/trailing '-' removed
-- chunk text = doc header + blank line + '## ' heading line + blank line + section body (binding Definitions)
CREATE OR REPLACE TEMP VIEW fresh_chunks AS
WITH docs AS (
  SELECT
    regexp_extract(path, '[^/]+$', 0) AS doc_name,
    TRIM(CAST(content AS STRING)) AS raw
  FROM read_files('/Volumes/claims/raw/landing/genai/policy_docs/', format => 'binaryFile')
),
parts AS (
  SELECT
    doc_name,
    TRIM(split(raw, '\\n## ')[0]) AS doc_header,
    pos,
    split(raw, '\\n## ')[pos] AS sec
  FROM docs
  LATERAL VIEW posexplode(split(raw, '\\n## ')) p AS pos, sec
  WHERE pos > 0
),
chunks AS (
  SELECT
    doc_name,
    -- Spark TRIM only strips spaces; use regex to also strip newlines (Definitions: exactly ONE blank line between parts)
    REGEXP_REPLACE(REGEXP_REPLACE(doc_header, '\\s+$', ''), '^\\s+', '') AS doc_header,
    pos - 1 AS chunk_index,
    REGEXP_REPLACE(REGEXP_REPLACE(split(sec, '\\n')[0], '\\s+$', ''), '^\\s+', '') AS heading,
    REGEXP_REPLACE(REGEXP_REPLACE(SUBSTRING(sec, POSITION('\n' IN sec) + 1), '\\s+$', ''), '^\\s+', '') AS body
  FROM parts
)
SELECT
  doc_name || '#' ||
    REGEXP_REPLACE(REGEXP_REPLACE(LOWER(heading), '[^a-z0-9]+', '-'), '^-+|-+$', '') AS chunk_id,
  doc_name,
  heading,
  CAST(chunk_index AS INT) AS chunk_index,
  CONCAT(doc_header, '\n\n', '## ', heading, '\n\n', body) AS text
FROM chunks;

-- uniqueness guard: MERGE requires a unique key on the source side
SELECT COUNT(*) AS src_chunks, COUNT(DISTINCT chunk_id) AS distinct_ids FROM fresh_chunks;

MERGE INTO claims.genai.policy_chunks AS t
USING fresh_chunks AS s
ON t.chunk_id = s.chunk_id
WHEN MATCHED AND t.text <> s.text
  THEN UPDATE SET
    t.doc_name = s.doc_name,
    t.heading = s.heading,
    t.chunk_index = s.chunk_index,
    t.text = s.text
WHEN NOT MATCHED
  THEN INSERT (chunk_id, doc_name, heading, chunk_index, text)
    VALUES (s.chunk_id, s.doc_name, s.heading, s.chunk_index, s.text)
WHEN NOT MATCHED BY SOURCE THEN DELETE;

In [0]:
%sql
-- R2 check: row counts + Delta history proof (>= 2 MERGE loads, last one 0 inserted / 0 deleted)
SELECT
  COUNT(*) AS chunks,
  COUNT(DISTINCT doc_name) AS docs,
  COUNT(DISTINCT chunk_id) AS distinct_ids
FROM claims.genai.policy_chunks;

SELECT
  version,
  timestamp,
  operation,
  operationMetrics
FROM (
  DESCRIBE HISTORY claims.genai.policy_chunks
)
WHERE operation IN ('MERGE', 'CREATE TABLE', 'CREATE TABLE AS SELECT')
ORDER BY version DESC
LIMIT 5;

In [0]:
# R2 Vector Search: endpoint claims_vs + Delta Sync index claims.genai.policy_chunks_idx
# Free Edition allows exactly ONE endpoint - claims_vs is reused by R4/R5. Idempotent: creates only what is missing.
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import vectorsearch

w = WorkspaceClient()
ENDPOINT_NAME = "claims_vs"
INDEX_NAME = f"{CATALOG}.genai.policy_chunks_idx"
SOURCE_TABLE = f"{CATALOG}.genai.policy_chunks"

# 1. Endpoint (create only if missing; Free Edition: exactly one)
eps = {e.name: e for e in w.vector_search_endpoints.list_endpoints()}
ep = eps.get(ENDPOINT_NAME)
if ep is None:
    print(f"Creating endpoint {ENDPOINT_NAME} ...")
    w.vector_search_endpoints.create_endpoint_and_wait(
        name=ENDPOINT_NAME, endpoint_type=vectorsearch.EndpointType.STANDARD)
    print("Endpoint ONLINE")
else:
    print(f"Reusing endpoint {ENDPOINT_NAME} ({ep.endpoint_status.state})")


In [0]:

# 2. Delta Sync index (TRIGGERED pipeline, pk chunk_id, embeddings from 'text' via EMBED_MODEL)
def get_index():
    try: 
        print(w.vector_search_indexes.get_index(index_name=INDEX_NAME))
        return w.vector_search_indexes.get_index(index_name=INDEX_NAME)
    except Exception:
        return None

idx = get_index()
if idx is None:
    print("Creating Delta Sync index ...")
    w.vector_search_indexes.create_index(
        name=INDEX_NAME,
        endpoint_name=ENDPOINT_NAME,
        primary_key="chunk_id",
        index_type=vectorsearch.VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=vectorsearch.DeltaSyncVectorIndexSpecRequest(
            source_table=SOURCE_TABLE,
            embedding_source_columns=[vectorsearch.EmbeddingSourceColumn(
                name="text", embedding_model_endpoint_name=EMBED_MODEL)],
            pipeline_type=vectorsearch.PipelineType.TRIGGERED))


In [0]:
st = w.vector_search_indexes.get_index(index_name=INDEX_NAME).status
print(st)

In [0]:

# 3. TRIGGERED pipeline: kick the initial sync if not ready, then poll until READY (all 35 chunks indexed)
# NOTE: this SDK's VectorIndexStatus exposes .ready (bool), .indexed_row_count, .message - no ready_status/detailed_status
idx = get_index()
if idx is not None and idx.status and not idx.status.ready:
    try:
        w.vector_search_indexes.sync_index(index_name=INDEX_NAME)
        print("Initial sync triggered ...")
    except Exception as e:
        print(f"Sync trigger skipped (still provisioning): {str(e)[:120]}")

for _ in range(80):
    idx = get_index()
    st = idx.status if idx else None
    ready = bool(st.ready) if st else False
    n = st.indexed_row_count if st else 0
    print(f"  ready={ready} indexed_rows={n} {('- ' + st.message[:60]) if st and st.message else ''}")
    if ready and n == 35:
        break
    time.sleep(15)

idx = get_index()
st = idx.status
print(f"FINAL: {INDEX_NAME} ready={st.ready} indexed={st.indexed_row_count} of 35")